In [ ]:
# ============================================================
# 06b_supervised_temporal_modelling
# Refactored supervised temporal validation pipeline
# ============================================================
#
# Purpose:
# - Train supervised classification models using reduced feature spaces.
# - Compare two clinically distinct workflows:
#
#   Workflow 1: Early prediction, days 0–3
#   Workflow 2: Full trajectory characterization, days 0–7
#
# - Preserve temporal validation:
#   Cohort 1 = training
#   Cohort 2 = temporal testing
#
# - Use RobustScaler for numerical variables.
# - Evaluate discrimination, PR-AUC, calibration, Brier score.
# - Generate interpretable plots and result summaries.
#
# ============================================================

In [ ]:
# ============================================================
# 01. IMPORTS
# ============================================================

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)

from sklearn.inspection import permutation_importance

from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)

In [ ]:
# ============================================================
# 02. PATHS
# ============================================================

PROJECT_CONSTANTS_DIR = Path(
    "/home/azureuser/cloudfiles/code/Users/m.bolognini/"
    "UseCase-Code/data-science/projects/a_MB_thesis_final/"
    "pipeline_thesis_final"
)

sys.path.append(str(PROJECT_CONSTANTS_DIR))

from project_constants import *

OUTPUT_05B = (
    PROJECT_CONSTANTS_DIR /
    "outputs" /
    "05b_feature_construction_and_selection"
)

OUTPUT_06B = (
    PROJECT_CONSTANTS_DIR /
    "outputs" /
    "06b_supervised_temporal_modelling"
)

OUTPUT_06B.mkdir(parents=True, exist_ok=True)

FEATURES_0_3_PATH = OUTPUT_05B / "05b_features_0_3.parquet"
FEATURES_0_7_PATH = OUTPUT_05B / "05b_features_0_7.parquet"

print("FEATURES_0_3_PATH:", FEATURES_0_3_PATH)
print("FEATURES_0_7_PATH:", FEATURES_0_7_PATH)
print("OUTPUT_06B:", OUTPUT_06B)

In [ ]:
# ============================================================
# 03. LOAD DATASETS
# ============================================================

df_0_3 = pd.read_parquet(FEATURES_0_3_PATH)
df_0_7 = pd.read_parquet(FEATURES_0_7_PATH)

print("0–3 dataset:", df_0_3.shape)
print("0–7 dataset:", df_0_7.shape)

display(df_0_3.head())
display(df_0_7.head())

In [ ]:
# ============================================================
# 04. OUTCOMES AND COHORTS
# ============================================================

OUTCOMES = {
    "composite_inhospital_or_palliative": "Composite in-hospital/palliative outcome",
    "mortality_90d_post_discharge": "90-day post-discharge mortality",
}

COHORT_DEFINITIONS = {
    "full": lambda df: df.index,
    "non_metastatic": lambda df: df.index[df["metastatic_group"] == "Non-metastatic"],
    "metastatic": lambda df: df.index[df["metastatic_group"] == "Metastatic"],
}

for name, df_current in [
    ("0_3", df_0_3),
    ("0_7", df_0_7),
]:
    print("\n" + "=" * 80)
    print("Dataset:", name)
    print("=" * 80)
    
    print("Temporal cohorts:")
    display(df_current["temporal_cohort"].value_counts(dropna=False))
    
    print("Metastatic groups:")
    display(df_current["metastatic_group"].value_counts(dropna=False))
    
    for outcome in OUTCOMES:
        print("\nOutcome:", outcome)
        display(df_current[outcome].value_counts(dropna=False))

In [ ]:
# Check missing clinical features from exported datasets

clinical_core_features = [
    "age",
    "sex",
    "bmi",
    "adm_mews",
    "admBarthelScore",
    "admBradenScore",
    "admECOG",
    "ccsi",
    "solidTumor_cat",
]

missing_0_3 = [
    c for c in clinical_core_features
    if c not in df_0_3.columns
]

missing_0_7 = [
    c for c in clinical_core_features
    if c not in df_0_7.columns
]

print("Missing from 0–3:", missing_0_3)
print("Missing from 0–7:", missing_0_7)

print("\nColumns containing 'Barthel', 'Braden', or 'ECOG':")
display([
    c for c in df_0_3.columns
    if "Barthel" in c or "Braden" in c or "ECOG" in c
])

In [ ]:
# ============================================================
# 05. DEFINE FEATURE GROUPS
# ============================================================

metadata_cols = [
    "episode_key",
    "pubID",
    "temporal_cohort",
    "metastatic_group",
    "hospitalWard",
    "endDate",
    "dateOfDeath",
]

metadata_cols_0_3 = [
    c for c in metadata_cols
    if c in df_0_3.columns
]

metadata_cols_0_7 = [
    c for c in metadata_cols
    if c in df_0_7.columns
]

outcome_cols = [
    c for c in OUTCOMES.keys()
    if c in df_0_3.columns or c in df_0_7.columns
]

clinical_core_features = [
    "age",
    "sex",
    "bmi",
    "adm_mews",
    "admBarthelScore",
    "admBradenScore",
    "admECOG",
    "ccsi",
    "solidTumor_cat",
]

clinical_core_0_3 = [
    c for c in clinical_core_features
    if c in df_0_3.columns
]

clinical_core_0_7 = [
    c for c in clinical_core_features
    if c in df_0_7.columns
]

lab_features_0_3 = [
    c for c in df_0_3.columns
    if (
        c not in metadata_cols_0_3
        and c not in outcome_cols
        and c not in clinical_core_0_3
    )
]

lab_features_0_7 = [
    c for c in df_0_7.columns
    if (
        c not in metadata_cols_0_7
        and c not in outcome_cols
        and c not in clinical_core_0_7
        and c != "LOS_days"
    )
]

print("Clinical 0–3:", len(clinical_core_0_3), clinical_core_0_3)
print("Lab trajectory 0–3:", len(lab_features_0_3), lab_features_0_3)

print("\nClinical 0–7:", len(clinical_core_0_7), clinical_core_0_7)
print("Lab trajectory 0–7:", len(lab_features_0_7), lab_features_0_7)

In [ ]:
# ============================================================
# 06. DEFINE MODELLING WORKFLOWS
# ============================================================

WORKFLOWS = {
    "early_0_3_prediction": {
        "df": df_0_3,
        "clinical_features": clinical_core_0_3,
        "lab_features": lab_features_0_3,
        "description": "Early prediction using features available during days 0–3",
    },
    "full_0_7_characterization": {
        "df": df_0_7,
        "clinical_features": clinical_core_0_7,
        "lab_features": lab_features_0_7,
        "description": "Full first-week trajectory characterization using days 0–7",
    },
}

for workflow_name, workflow in WORKFLOWS.items():
    print("\n" + "=" * 80)
    print(workflow_name)
    print("=" * 80)
    print(workflow["description"])
    print("Clinical features:", len(workflow["clinical_features"]))
    print("Lab/trajectory features:", len(workflow["lab_features"]))

In [ ]:
# ============================================================
# 07. DEFINE FEATURE SETS
# ============================================================

def get_feature_sets(workflow):
    clinical = workflow["clinical_features"]
    labs = workflow["lab_features"]

    return {
        "Model_A_clinical": clinical,
        "Model_B_clinical_labs_trajectory": clinical + labs,
        "Model_C_labs_trajectory_only": labs,
    }


for workflow_name, workflow in WORKFLOWS.items():
    feature_sets = get_feature_sets(workflow)

    print("\n" + "=" * 80)
    print(workflow_name)
    print("=" * 80)

    for feature_set_name, features in feature_sets.items():
        features = list(dict.fromkeys(features))
        print(feature_set_name, len(features))

In [ ]:
# ============================================================
# 08. DEFINE MODELS
# ============================================================

BASE_MODELS = {
    "Logistic_L2": LogisticRegression(
        penalty="l2",
        solver="liblinear",
        class_weight="balanced",
        max_iter=3000,
        random_state=42,
    ),
    "Logistic_L1": LogisticRegression(
        penalty="l1",
        solver="liblinear",
        class_weight="balanced",
        max_iter=3000,
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        max_depth=4,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=500,
        max_depth=4,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=200,
        max_leaf_nodes=15,
        l2_regularization=0.1,
        random_state=42,
    ),
}

print(BASE_MODELS.keys())

In [ ]:
# ============================================================
# 09. CLEANING AND PREPROCESSING HELPERS
# ============================================================

def clean_X_for_model(X):
    X = X.copy()
    X = X.loc[:, ~X.columns.duplicated()].copy()
    
    for col in X.columns:
        if X[col].dtype == "bool":
            X[col] = X[col].astype(int)
        elif (
            X[col].dtype == "object"
            or str(X[col].dtype) in ["string", "category"]
        ):
            X[col] = (
                X[col]
                .astype("string")
                .fillna("Missing")
                .astype(str)
            )
        else:
            X[col] = pd.to_numeric(X[col], errors="coerce")
    
    return X


def get_preprocessor(X):
    X = clean_X_for_model(X)
    
    numeric_features = X.select_dtypes(
        include=["number"]
    ).columns.tolist()
    
    categorical_features = [
        c for c in X.columns
        if c not in numeric_features
    ]
    
    numeric_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            ("scaler", RobustScaler()),
        ]
    )
    
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop",
    )
    
    return preprocessor


def get_feature_names_from_preprocessor(preprocessor):
    feature_names = []
    
    try:
        num_pipeline = preprocessor.named_transformers_["num"]
        cat_pipeline = preprocessor.named_transformers_["cat"]
        
        num_features = preprocessor.transformers_[0][2]
        cat_features = preprocessor.transformers_[1][2]
        
        num_imputer = num_pipeline.named_steps["imputer"]
        
        numeric_output_names = list(num_features)
        
        if hasattr(num_imputer, "indicator_"):
            indicator_features = [
                f"{num_features[i]}_missing_indicator"
                for i in num_imputer.indicator_.features_
            ]
            numeric_output_names += indicator_features
        
        cat_encoder = cat_pipeline.named_steps["onehot"]
        
        if len(cat_features) > 0:
            cat_output_names = list(
                cat_encoder.get_feature_names_out(cat_features)
            )
        else:
            cat_output_names = []
        
        feature_names = numeric_output_names + cat_output_names
    
    except Exception:
        feature_names = []
    
    return feature_names

In [ ]:
# ============================================================
# 10. METRIC HELPERS
# ============================================================

def calibration_intercept_slope(y_true, y_prob):
    eps = 1e-6
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    logit = np.log(
        y_prob / (1 - y_prob)
    ).reshape(-1, 1)
    
    try:
        cal_model = LogisticRegression(
            penalty=None,
            solver="lbfgs",
            max_iter=1000,
        )
        cal_model.fit(logit, y_true)
        
        return (
            cal_model.intercept_[0],
            cal_model.coef_[0][0],
        )
    
    except Exception:
        return np.nan, np.nan


def compute_metrics(y_true, y_prob):
    out = {
        "n": len(y_true),
        "events": int(np.sum(y_true)),
        "event_rate": float(np.mean(y_true)),
    }
    
    if len(np.unique(y_true)) < 2:
        out.update({
            "roc_auc": np.nan,
            "pr_auc": np.nan,
            "brier": np.nan,
            "calibration_intercept": np.nan,
            "calibration_slope": np.nan,
        })
        return out
    
    out["roc_auc"] = roc_auc_score(y_true, y_prob)
    out["pr_auc"] = average_precision_score(y_true, y_prob)
    out["brier"] = brier_score_loss(y_true, y_prob)
    
    intercept, slope = calibration_intercept_slope(
        y_true,
        y_prob,
    )
    
    out["calibration_intercept"] = intercept
    out["calibration_slope"] = slope
    
    return out


def bootstrap_metric_ci(
    y_true,
    y_prob,
    metric_fn,
    n_bootstrap=500,
    seed=42,
):
    rng = np.random.default_rng(seed)
    
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    
    values = []
    n = len(y_true)
    
    for _ in range(n_bootstrap):
        idx = rng.choice(
            np.arange(n),
            size=n,
            replace=True,
        )
        
        y_b = y_true[idx]
        p_b = y_prob[idx]
        
        if len(np.unique(y_b)) < 2:
            continue
        
        try:
            values.append(metric_fn(y_b, p_b))
        except Exception:
            continue
    
    if len(values) == 0:
        return np.nan, np.nan
    
    return (
        np.percentile(values, 2.5),
        np.percentile(values, 97.5),
    )


def add_bootstrap_ci_to_metrics(
    metrics,
    y_true,
    y_prob,
    n_bootstrap=500,
):
    roc_low, roc_high = bootstrap_metric_ci(
        y_true,
        y_prob,
        roc_auc_score,
        n_bootstrap=n_bootstrap,
    )
    
    pr_low, pr_high = bootstrap_metric_ci(
        y_true,
        y_prob,
        average_precision_score,
        n_bootstrap=n_bootstrap,
    )
    
    metrics["roc_auc_ci_low"] = roc_low
    metrics["roc_auc_ci_high"] = roc_high
    metrics["pr_auc_ci_low"] = pr_low
    metrics["pr_auc_ci_high"] = pr_high
    
    return metrics

In [ ]:
# ============================================================
# 11. RISK GROUP HELPER
# ============================================================

def assign_train_based_tertiles(train_prob, eval_prob):
    q1, q2 = np.quantile(
        train_prob,
        [1 / 3, 2 / 3],
    )
    
    def assign(x):
        if x <= q1:
            return "Low"
        elif x <= q2:
            return "Intermediate"
        else:
            return "High"
    
    return pd.Series(eval_prob).apply(assign).values


def summarize_risk_groups(y_true, y_prob, risk_group):
    risk_df = pd.DataFrame({
        "y_true": y_true,
        "y_prob": y_prob,
        "risk_group": risk_group,
    })
    
    summary = (
        risk_df
        .groupby("risk_group", observed=True)
        .agg(
            n=("y_true", "size"),
            events=("y_true", "sum"),
            observed_event_rate=("y_true", "mean"),
            mean_predicted_risk=("y_prob", "mean"),
            median_predicted_risk=("y_prob", "median"),
        )
        .reset_index()
    )
    
    return summary

In [ ]:
# ============================================================
# 12. PIPELINE BUILDER
# ============================================================

def build_pipeline(X_train, classifier):
    preprocessor = get_preprocessor(X_train)
    
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", clone(classifier)),
        ]
    )
    
    return pipe

In [ ]:
# ============================================================
# 13. PLOT HELPERS
# ============================================================

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace(" ", "_")
        .replace("|", "_")
        .replace(":", "_")
    )


def plot_roc_curve(y_train, p_train, y_test, p_test, title, output_path=None):
    plt.figure(figsize=(6, 5))
    
    for y, p, label in [
        (y_train, p_train, "Train"),
        (y_test, p_test, "Test"),
    ]:
        if len(np.unique(y)) < 2:
            continue
        
        fpr, tpr, _ = roc_curve(y, p)
        auc = roc_auc_score(y, p)
        
        plt.plot(
            fpr,
            tpr,
            label=f"{label} AUC={auc:.3f}",
        )
    
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    
    if output_path is not None:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
    
    plt.show()


def plot_pr_curve(y_train, p_train, y_test, p_test, title, output_path=None):
    plt.figure(figsize=(6, 5))
    
    for y, p, label in [
        (y_train, p_train, "Train"),
        (y_test, p_test, "Test"),
    ]:
        if len(np.unique(y)) < 2:
            continue
        
        precision, recall, _ = precision_recall_curve(y, p)
        ap = average_precision_score(y, p)
        
        plt.plot(
            recall,
            precision,
            label=f"{label} AP={ap:.3f}",
        )
    
    baseline = np.mean(y)
    plt.axhline(
        baseline,
        linestyle="--",
        label=f"{label} baseline={baseline:.3f}"
   )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    
    if output_path is not None:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
    
    plt.show()


def plot_calibration_curve(y_train, p_train, y_test, p_test, title, output_path=None):
    plt.figure(figsize=(6, 5))
    
    for y, p, label in [
        (y_train, p_train, "Train"),
        (y_test, p_test, "Test"),
    ]:
        if len(np.unique(y)) < 2:
            continue
        
        try:
            bins = pd.qcut(
                p,
                q=5,
                duplicates="drop",
            )
            
            cal = (
                pd.DataFrame({
                    "y": y,
                    "p": p,
                    "bin": bins,
                })
                .groupby("bin", observed=True)
                .agg(
                    mean_pred=("p", "mean"),
                    obs_rate=("y", "mean"),
                    n=("y", "size"),
                )
                .reset_index()
            )
            
            plt.plot(
                cal["mean_pred"],
                cal["obs_rate"],
                marker="o",
                label=label,
            )
        
        except Exception:
            continue
    
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Mean predicted risk")
    plt.ylabel("Observed event rate")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    
    if output_path is not None:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
    
    plt.show()

In [ ]:
# ============================================================
# 13b. MODEL SELECTION STRATEGY
# ============================================================

SELECTION_METRIC = "pr_auc"
VALIDATION_SIZE = 0.25
RANDOM_STATE = 42

print("Model selection metric:", SELECTION_METRIC)
print("Internal validation size:", VALIDATION_SIZE)

print("""
Model selection strategy:
- Cohort 1 is split into internal train and validation sets.
- Models are fitted on the internal train set.
- Best models are selected using validation PR-AUC.
- Selected models are then refitted on the full Cohort 1.
- Final performance is evaluated once on Cohort 2.
""")

In [ ]:
# ============================================================
# 14. MAIN TEMPORAL VALIDATION LOOP
# ============================================================

results = []
predictions = []
risk_group_rows = []
best_models = {}

for workflow_name, workflow in WORKFLOWS.items():

    df_workflow = workflow["df"].copy()
    feature_sets = get_feature_sets(workflow)

    print("\n" + "#" * 100)
    print("WORKFLOW:", workflow_name)
    print(workflow["description"])
    print("#" * 100)

    for cohort_name, cohort_fn in COHORT_DEFINITIONS.items():

        cohort_idx = cohort_fn(df_workflow)
        df_sub = df_workflow.loc[cohort_idx].copy()

        for outcome_col, outcome_label in OUTCOMES.items():

            if outcome_col not in df_sub.columns:
                continue

            df_out = df_sub[df_sub[outcome_col].notna()].copy()

            cohort1_df = df_out[
                df_out["temporal_cohort"] == "Cohort 1"
            ].copy()

            test_df = df_out[
                df_out["temporal_cohort"] == "Cohort 2"
            ].copy()

            if cohort1_df.empty or test_df.empty:
                continue

            y_cohort1 = cohort1_df[outcome_col].astype(int).values
            y_test = test_df[outcome_col].astype(int).values

            if len(np.unique(y_cohort1)) < 2 or len(np.unique(y_test)) < 2:
                continue

            # ------------------------------------------------------------
            # Internal train/validation split within Cohort 1
            # ------------------------------------------------------------
            train_df, val_df = train_test_split(
                cohort1_df,
                test_size=VALIDATION_SIZE,
                stratify=cohort1_df[outcome_col].astype(int),
                random_state=RANDOM_STATE,
            )

            y_train = train_df[outcome_col].astype(int).values
            y_val = val_df[outcome_col].astype(int).values

            print("\n" + "=" * 80)
            print(workflow_name, "|", cohort_name, "|", outcome_col)
            print("Internal train:", len(train_df), "events:", y_train.sum())
            print("Internal val  :", len(val_df), "events:", y_val.sum())
            print("Temporal test :", len(test_df), "events:", y_test.sum())

            key = (
                workflow_name,
                cohort_name,
                outcome_col,
            )

            best_models[key] = {
                "val_pr_auc": -np.inf
            }

            for feature_set_name, features in feature_sets.items():

                features = list(dict.fromkeys([
                    f for f in features
                    if f in df_out.columns
                ]))

                X_train = clean_X_for_model(
                    train_df[features].copy()
                )

                X_val = clean_X_for_model(
                    val_df[features].copy()
                )

                X_test = clean_X_for_model(
                    test_df[features].copy()
                )

                X_val = X_val[X_train.columns]
                X_test = X_test[X_train.columns]

                for model_name, classifier in BASE_MODELS.items():

                    pipe = build_pipeline(
                        X_train=X_train,
                        classifier=classifier,
                    )

                    try:
                        # Fit on internal train only
                        pipe.fit(X_train, y_train)

                        p_train = pipe.predict_proba(X_train)[:, 1]
                        p_val = pipe.predict_proba(X_val)[:, 1]
                        p_test_pre_refit = pipe.predict_proba(X_test)[:, 1]

                    except Exception as e:
                        print(
                            "FAILED:",
                            workflow_name,
                            cohort_name,
                            outcome_col,
                            feature_set_name,
                            model_name,
                            e,
                        )
                        continue

                    train_metrics = compute_metrics(
                        y_train,
                        p_train,
                    )

                    val_metrics = compute_metrics(
                        y_val,
                        p_val,
                    )

                    test_pre_refit_metrics = compute_metrics(
                        y_test,
                        p_test_pre_refit,
                    )

                    row = {
                        "workflow": workflow_name,
                        "cohort": cohort_name,
                        "outcome": outcome_col,
                        "outcome_label": outcome_label,
                        "feature_set": feature_set_name,
                        "model_name": model_name,
                        "n_features_original": len(features),
                        "selection_metric": "val_pr_auc",
                    }

                    for k, v in train_metrics.items():
                        row[f"internal_train_{k}"] = v

                    for k, v in val_metrics.items():
                        row[f"val_{k}"] = v

                    for k, v in test_pre_refit_metrics.items():
                        row[f"test_pre_refit_{k}"] = v

                    results.append(row)

                    # Select best model using internal validation PR-AUC
                    if val_metrics["pr_auc"] > best_models[key]["val_pr_auc"]:
                        best_models[key] = {
                            "val_pr_auc": val_metrics["pr_auc"],
                            "pipe": pipe,
                            "feature_set": feature_set_name,
                            "model_name": model_name,
                            "features": features,
                            "X_train_internal": X_train,
                            "X_val": X_val,
                            "X_test": X_test,
                            "y_train_internal": y_train,
                            "y_val": y_val,
                            "y_test": y_test,
                            "p_train_internal": p_train,
                            "p_val": p_val,
                            "p_test_pre_refit": p_test_pre_refit,
                            "train_df_internal": train_df.copy(),
                            "val_df": val_df.copy(),
                            "test_df": test_df.copy(),
                        }

            # ------------------------------------------------------------
            # Refit selected model on full Cohort 1 and evaluate on Cohort 2
            # ------------------------------------------------------------
            selected = best_models[key]

            if "features" not in selected:
                continue

            features = selected["features"]
            classifier = BASE_MODELS[selected["model_name"]]

            X_cohort1_full = clean_X_for_model(
                cohort1_df[features].copy()
            )

            X_test_final = clean_X_for_model(
                test_df[features].copy()
            )

            X_test_final = X_test_final[X_cohort1_full.columns]

            final_pipe = build_pipeline(
                X_train=X_cohort1_full,
                classifier=classifier,
            )

            final_pipe.fit(
                X_cohort1_full,
                y_cohort1,
            )

            p_cohort1_full = final_pipe.predict_proba(X_cohort1_full)[:, 1]
            p_test_final = final_pipe.predict_proba(X_test_final)[:, 1]

            cohort1_metrics = compute_metrics(
                y_cohort1,
                p_cohort1_full,
            )

            test_metrics = compute_metrics(
                y_test,
                p_test_final,
            )

            test_metrics = add_bootstrap_ci_to_metrics(
                test_metrics,
                y_test,
                p_test_final,
                n_bootstrap=500,
            )

            selected.update({
                "final_pipe": final_pipe,
                "X_cohort1_full": X_cohort1_full,
                "X_test_final": X_test_final,
                "y_cohort1_full": y_cohort1,
                "p_cohort1_full": p_cohort1_full,
                "p_test_final": p_test_final,
                "cohort1_df": cohort1_df.copy(),
                "test_df": test_df.copy(),
                "cohort1_metrics": cohort1_metrics,
                "test_metrics": test_metrics,
            })

            # Save final train/test predictions after refit
            pred_train_df = pd.DataFrame({
                "episode_key": cohort1_df["episode_key"].values,
                "pubID": cohort1_df["pubID"].values,
                "workflow": workflow_name,
                "cohort": cohort_name,
                "outcome": outcome_col,
                "feature_set": selected["feature_set"],
                "model_name": selected["model_name"],
                "split": "cohort1_refit_train",
                "y_true": y_cohort1,
                "y_prob": p_cohort1_full,
                "temporal_cohort": cohort1_df["temporal_cohort"].values,
                "metastatic_group": cohort1_df["metastatic_group"].values,
            })

            pred_test_df = pd.DataFrame({
                "episode_key": test_df["episode_key"].values,
                "pubID": test_df["pubID"].values,
                "workflow": workflow_name,
                "cohort": cohort_name,
                "outcome": outcome_col,
                "feature_set": selected["feature_set"],
                "model_name": selected["model_name"],
                "split": "cohort2_temporal_test",
                "y_true": y_test,
                "y_prob": p_test_final,
                "temporal_cohort": test_df["temporal_cohort"].values,
                "metastatic_group": test_df["metastatic_group"].values,
            })

            predictions.append(pred_train_df)
            predictions.append(pred_test_df)

            train_groups = assign_train_based_tertiles(
                train_prob=p_cohort1_full,
                eval_prob=p_cohort1_full,
            )

            test_groups = assign_train_based_tertiles(
                train_prob=p_cohort1_full,
                eval_prob=p_test_final,
            )

            risk_summary = summarize_risk_groups(
                y_true=y_test,
                y_prob=p_test_final,
                risk_group=test_groups,
            )

            for _, rr in risk_summary.iterrows():
                risk_group_rows.append({
                    "workflow": workflow_name,
                    "cohort": cohort_name,
                    "outcome": outcome_col,
                    "feature_set": selected["feature_set"],
                    "model_name": selected["model_name"],
                    "risk_group": rr["risk_group"],
                    "n": rr["n"],
                    "events": rr["events"],
                    "observed_event_rate": rr["observed_event_rate"],
                    "mean_predicted_risk": rr["mean_predicted_risk"],
                    "median_predicted_risk": rr["median_predicted_risk"],
                })


results_df = pd.DataFrame(results)
predictions_df = pd.concat(predictions, ignore_index=True)
risk_groups_df = pd.DataFrame(risk_group_rows)

display(results_df.head())
display(risk_groups_df.head())

In [ ]:
# ============================================================
# 15. BEST MODEL SUMMARY
# ============================================================

best_rows = []

for key, obj in best_models.items():

    workflow_name, cohort_name, outcome_col = key

    if "test_metrics" not in obj:
        continue

    row = {
        "workflow": workflow_name,
        "cohort": cohort_name,
        "outcome": outcome_col,
        "feature_set": obj["feature_set"],
        "model_name": obj["model_name"],
        "n_features_original": len(obj["features"]),
        "selection_metric": "val_pr_auc",
        "val_pr_auc": obj["val_pr_auc"],
    }

    for k, v in obj["cohort1_metrics"].items():
        row[f"train_{k}"] = v

    for k, v in obj["test_metrics"].items():
        row[f"test_{k}"] = v

    best_rows.append(row)

best_summary = pd.DataFrame(best_rows)

display(
    best_summary.sort_values(
        ["workflow", "cohort", "outcome"]
    )
)

In [ ]:
plt.hist(p_test_final, bins=20)
plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.title("Distribution of predicted probabilities")
plt.show()

In [ ]:
# ============================================================
# 16. BEST MODEL ROC, PR, AND CALIBRATION PLOTS
# ============================================================

for key, obj in best_models.items():
    workflow_name, cohort_name, outcome_col = key
    
    title_base = (
        f"{workflow_name} | {cohort_name} | {outcome_col}\n"
        f"{obj['feature_set']} | {obj['model_name']}"
    )
    
    output_prefix = safe_name(
        f"{workflow_name}_{cohort_name}_{outcome_col}_"
        f"{obj['feature_set']}_{obj['model_name']}"
    )
    
    plot_roc_curve(
    y_train=obj["y_cohort1_full"],
    p_train=obj["p_cohort1_full"],
    y_test=obj["y_test"],
    p_test=obj["p_test_final"],
    title=f"ROC curve — {title_base}",
    output_path=OUTPUT_06B / f"{output_prefix}_roc.png",
    )
    
    plot_pr_curve(
        y_train=obj["y_cohort1_full"],
        p_train=obj["p_cohort1_full"],
        y_test=obj["y_test"],
        p_test=obj["p_test_final"],
        title=f"Precision–Recall curve — {title_base}",
        output_path=OUTPUT_06B / f"{output_prefix}_pr.png",
    )
    
    plot_calibration_curve(
        y_train=obj["y_cohort1_full"],
        p_train=obj["p_cohort1_full"],
        y_test=obj["y_test"],
        p_test=obj["p_test_final"],
        title=f"Calibration — {title_base}",
        output_path=OUTPUT_06B / f"{output_prefix}_calibration.png",
    )

In [ ]:
# ============================================================
# 17. BEST MODEL WITHIN EACH FEATURE SET
# Selected using internal validation PR-AUC
# ============================================================

selection_metric = "val_pr_auc"

best_by_feature_set = (
    results_df
    .sort_values(selection_metric, ascending=False)
    .groupby(
        [
            "workflow",
            "cohort",
            "outcome",
            "feature_set",
        ],
        as_index=False,
    )
    .first()
)

cols_to_display = [
    "workflow",
    "cohort",
    "outcome",
    "feature_set",
    "model_name",
    "n_features_original",
    "selection_metric",
    "internal_train_n",
    "internal_train_events",
    "internal_train_event_rate",
    "val_n",
    "val_events",
    "val_event_rate",
    "val_roc_auc",
    "val_pr_auc",
    "val_brier",
    "val_calibration_slope",
    "test_pre_refit_n",
    "test_pre_refit_events",
    "test_pre_refit_event_rate",
    "test_pre_refit_roc_auc",
    "test_pre_refit_pr_auc",
    "test_pre_refit_brier",
    "test_pre_refit_calibration_slope",
]

cols_to_display = [
    c for c in cols_to_display
    if c in best_by_feature_set.columns
]

display(
    best_by_feature_set[cols_to_display]
    .sort_values(["workflow", "cohort", "outcome", "feature_set"])
)

In [ ]:
# ============================================================
# 17b. BEST MODEL BY FEATURE SET — REFIT ON FULL COHORT 1
# ============================================================

best_by_feature_set_refit_rows = []
best_by_feature_set_refit_predictions = []

for _, selected_row in best_by_feature_set.iterrows():

    workflow_name = selected_row["workflow"]
    cohort_name = selected_row["cohort"]
    outcome_col = selected_row["outcome"]
    feature_set_name = selected_row["feature_set"]
    model_name = selected_row["model_name"]

    workflow = WORKFLOWS[workflow_name]
    df_workflow = workflow["df"].copy()
    feature_sets = get_feature_sets(workflow)

    cohort_idx = COHORT_DEFINITIONS[cohort_name](df_workflow)
    df_sub = df_workflow.loc[cohort_idx].copy()

    df_out = df_sub[df_sub[outcome_col].notna()].copy()

    cohort1_df = df_out[
        df_out["temporal_cohort"] == "Cohort 1"
    ].copy()

    test_df = df_out[
        df_out["temporal_cohort"] == "Cohort 2"
    ].copy()

    if cohort1_df.empty or test_df.empty:
        continue

    y_cohort1 = cohort1_df[outcome_col].astype(int).values
    y_test = test_df[outcome_col].astype(int).values

    if len(np.unique(y_cohort1)) < 2 or len(np.unique(y_test)) < 2:
        continue

    features = list(dict.fromkeys([
        f for f in feature_sets[feature_set_name]
        if f in df_out.columns
    ]))

    X_cohort1 = clean_X_for_model(
        cohort1_df[features].copy()
    )

    X_test = clean_X_for_model(
        test_df[features].copy()
    )

    X_test = X_test[X_cohort1.columns]

    classifier = BASE_MODELS[model_name]

    final_pipe = build_pipeline(
        X_train=X_cohort1,
        classifier=classifier,
    )

    try:
        final_pipe.fit(X_cohort1, y_cohort1)

        p_cohort1 = final_pipe.predict_proba(X_cohort1)[:, 1]
        p_test = final_pipe.predict_proba(X_test)[:, 1]

    except Exception as e:
        print(
            "Refit failed:",
            workflow_name,
            cohort_name,
            outcome_col,
            feature_set_name,
            model_name,
            e,
        )
        continue

    train_metrics = compute_metrics(
        y_cohort1,
        p_cohort1,
    )

    test_metrics = compute_metrics(
        y_test,
        p_test,
    )

    test_metrics = add_bootstrap_ci_to_metrics(
        test_metrics,
        y_test,
        p_test,
        n_bootstrap=500,
    )

    row = {
        "workflow": workflow_name,
        "cohort": cohort_name,
        "outcome": outcome_col,
        "feature_set": feature_set_name,
        "model_name": model_name,
        "n_features_original": len(features),
        "selection_metric": "val_pr_auc",
        "val_pr_auc": selected_row["val_pr_auc"],
    }

    for k, v in train_metrics.items():
        row[f"train_{k}"] = v

    for k, v in test_metrics.items():
        row[f"test_{k}"] = v

    best_by_feature_set_refit_rows.append(row)

    pred_test = pd.DataFrame({
        "episode_key": test_df["episode_key"].values,
        "pubID": test_df["pubID"].values,
        "workflow": workflow_name,
        "cohort": cohort_name,
        "outcome": outcome_col,
        "feature_set": feature_set_name,
        "model_name": model_name,
        "split": "cohort2_temporal_test",
        "y_true": y_test,
        "y_prob": p_test,
        "temporal_cohort": test_df["temporal_cohort"].values,
        "metastatic_group": test_df["metastatic_group"].values,
    })

    best_by_feature_set_refit_predictions.append(pred_test)

best_by_feature_set_refit = pd.DataFrame(
    best_by_feature_set_refit_rows
)

best_by_feature_set_refit_predictions = pd.concat(
    best_by_feature_set_refit_predictions,
    ignore_index=True,
)

display(
    best_by_feature_set_refit
    .sort_values(["workflow", "cohort", "outcome", "feature_set"])
)

In [ ]:
# ============================================================
# 18. FEATURE SET COMPARISON
# Final A/B/C comparison after refit on full Cohort 1
# ============================================================

comparison_rows = []

for keys, tmp in best_by_feature_set_refit.groupby(
    ["workflow", "cohort", "outcome"]
):

    workflow_name, cohort_name, outcome_col = keys
    tmp = tmp.copy()

    model_a = tmp[tmp["feature_set"] == "Model_A_clinical"]
    model_b = tmp[tmp["feature_set"] == "Model_B_clinical_labs_trajectory"]
    model_c = tmp[tmp["feature_set"] == "Model_C_labs_trajectory_only"]

    row = {
        "workflow": workflow_name,
        "cohort": cohort_name,
        "outcome": outcome_col,
    }

    if not model_a.empty:
        model_a = model_a.iloc[0]
        row.update({
            "model_A_algorithm": model_a["model_name"],
            "model_A_roc_auc": model_a["test_roc_auc"],
            "model_A_pr_auc": model_a["test_pr_auc"],
            "model_A_brier": model_a["test_brier"],
        })

    if not model_b.empty:
        model_b = model_b.iloc[0]
        row.update({
            "model_B_algorithm": model_b["model_name"],
            "model_B_roc_auc": model_b["test_roc_auc"],
            "model_B_pr_auc": model_b["test_pr_auc"],
            "model_B_brier": model_b["test_brier"],
        })

    if not model_c.empty:
        model_c = model_c.iloc[0]
        row.update({
            "model_C_algorithm": model_c["model_name"],
            "model_C_roc_auc": model_c["test_roc_auc"],
            "model_C_pr_auc": model_c["test_pr_auc"],
            "model_C_brier": model_c["test_brier"],
        })

    if "model_A_pr_auc" in row and "model_B_pr_auc" in row:
        row["delta_roc_auc_B_minus_A"] = (
            row["model_B_roc_auc"] - row["model_A_roc_auc"]
        )
        row["delta_pr_auc_B_minus_A"] = (
            row["model_B_pr_auc"] - row["model_A_pr_auc"]
        )
        row["delta_brier_B_minus_A"] = (
            row["model_B_brier"] - row["model_A_brier"]
        )

    if "model_A_pr_auc" in row and "model_C_pr_auc" in row:
        row["delta_roc_auc_C_minus_A"] = (
            row["model_C_roc_auc"] - row["model_A_roc_auc"]
        )
        row["delta_pr_auc_C_minus_A"] = (
            row["model_C_pr_auc"] - row["model_A_pr_auc"]
        )
        row["delta_brier_C_minus_A"] = (
            row["model_C_brier"] - row["model_A_brier"]
        )

    if "model_B_pr_auc" in row and "model_C_pr_auc" in row:
        row["delta_pr_auc_B_minus_C"] = (
            row["model_B_pr_auc"] - row["model_C_pr_auc"]
        )
        row["delta_roc_auc_B_minus_C"] = (
            row["model_B_roc_auc"] - row["model_C_roc_auc"]
        )

    comparison_rows.append(row)

model_comparison = pd.DataFrame(comparison_rows)

display(
    model_comparison.sort_values(
        ["workflow", "cohort", "outcome"]
    )
)

In [ ]:
# ============================================================
# 19. INTERPRETATIVE PLOT — TEST ROC-AUC
# ============================================================

plot_df = best_summary.copy()

plot_df["plot_label"] = (
    plot_df["workflow"] + "\n" +
    plot_df["cohort"] + " | " +
    plot_df["outcome"]
)

plot_df = plot_df.sort_values("test_roc_auc")

plt.figure(figsize=(10, 8))

plt.barh(
    plot_df["plot_label"],
    plot_df["test_roc_auc"],
)

plt.axvline(0.5, linestyle="--")
plt.axvline(0.7, linestyle="--")

plt.xlabel("Test ROC-AUC")
plt.title("Best model temporal test discrimination")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 20. INTERPRETATIVE PLOT — ROC-AUC WITH CI
# ============================================================

plot_df = best_summary.copy()

plot_df["plot_label"] = (
    plot_df["workflow"] + "\n" +
    plot_df["cohort"] + " | " +
    plot_df["outcome"]
)

plot_df = plot_df.sort_values("test_roc_auc")

x = plot_df["test_roc_auc"]
xerr_low = x - plot_df["test_roc_auc_ci_low"]
xerr_high = plot_df["test_roc_auc_ci_high"] - x

plt.figure(figsize=(10, 8))

plt.errorbar(
    x=x,
    y=np.arange(len(plot_df)),
    xerr=[xerr_low, xerr_high],
    fmt="o",
    capsize=4,
)

plt.yticks(np.arange(len(plot_df)), plot_df["plot_label"])

plt.axvline(0.5, linestyle="--")
plt.axvline(0.7, linestyle="--")

plt.xlabel("Test ROC-AUC with 95% bootstrap CI")
plt.title("Uncertainty of temporal test ROC-AUC")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 21. INTERPRETATIVE PLOT — PR-AUC LIFT OVER EVENT RATE
# ============================================================

plot_df = best_summary.copy()

plot_df["pr_auc_lift_over_event_rate"] = (
    plot_df["test_pr_auc"] / plot_df["test_event_rate"]
)

plot_df["plot_label"] = (
    plot_df["workflow"] + "\n" +
    plot_df["cohort"] + " | " +
    plot_df["outcome"]
)

plot_df = plot_df.sort_values("pr_auc_lift_over_event_rate")

plt.figure(figsize=(10, 8))

plt.barh(
    plot_df["plot_label"],
    plot_df["pr_auc_lift_over_event_rate"],
)

plt.axvline(1, linestyle="--")

plt.xlabel("PR-AUC / event rate")
plt.title("Precision–recall lift over random baseline")

plt.tight_layout()
plt.show()

display(
    plot_df[
        [
            "workflow",
            "cohort",
            "outcome",
            "test_event_rate",
            "test_pr_auc",
            "pr_auc_lift_over_event_rate",
        ]
    ].sort_values("pr_auc_lift_over_event_rate", ascending=False)
)

In [ ]:
# ============================================================
# 22. INTERPRETATIVE PLOT — DELTA PR-AUC VS CLINICAL MODEL
# Final refit-based comparison
# ============================================================

plot_df = model_comparison.copy()

plot_df["plot_label"] = (
    plot_df["workflow"] + "\n" +
    plot_df["cohort"] + " | " +
    plot_df["outcome"]
)

plot_df = plot_df[
    plot_df["delta_pr_auc_B_minus_A"].notna()
].copy()

plot_df = plot_df.sort_values("delta_pr_auc_B_minus_A")

plt.figure(figsize=(10, 7))

plt.barh(
    plot_df["plot_label"],
    plot_df["delta_pr_auc_B_minus_A"],
    label="Model B - Model A",
)

if "delta_pr_auc_C_minus_A" in plot_df.columns:
    plt.scatter(
        plot_df["delta_pr_auc_C_minus_A"],
        plot_df["plot_label"],
        marker="o",
        label="Model C - Model A",
    )

plt.axvline(0, linestyle="--")

plt.xlabel("Δ PR-AUC vs clinical model")
plt.title("Incremental value of laboratory trajectory features")

plt.legend()
plt.tight_layout()
plt.show()

display(
    plot_df[
        [
            "workflow",
            "cohort",
            "outcome",
            "model_A_pr_auc",
            "model_B_pr_auc",
            "model_C_pr_auc",
            "delta_pr_auc_B_minus_A",
            "delta_pr_auc_C_minus_A",
            "delta_pr_auc_B_minus_C",
        ]
    ].sort_values("delta_pr_auc_B_minus_A", ascending=False)
)

In [ ]:
# ============================================================
# 23. INTERPRETATIVE PLOT — GENERALIZATION GAP
# Internal train vs validation gap
# ============================================================

results_df["roc_gap_train_minus_val"] = (
    results_df["internal_train_roc_auc"] - results_df["val_roc_auc"]
)

results_df["pr_gap_train_minus_val"] = (
    results_df["internal_train_pr_auc"] - results_df["val_pr_auc"]
)

gap_summary = (
    results_df
    .groupby("model_name", as_index=False)
    .agg(
        mean_roc_gap=("roc_gap_train_minus_val", "mean"),
        median_roc_gap=("roc_gap_train_minus_val", "median"),
        mean_pr_gap=("pr_gap_train_minus_val", "mean"),
        median_pr_gap=("pr_gap_train_minus_val", "median"),
    )
    .sort_values("mean_roc_gap")
)

display(gap_summary)

plt.figure(figsize=(8, 5))

plt.barh(
    gap_summary["model_name"],
    gap_summary["mean_roc_gap"],
)

plt.axvline(0, linestyle="--")

plt.xlabel("Mean internal train-validation ROC-AUC gap")
plt.title("Average overfitting by model type")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 24. INTERPRETATIVE PLOT — SAMPLE SIZE VS UNCERTAINTY
# ============================================================

plot_df = best_summary.copy()

plot_df["roc_auc_ci_width"] = (
    plot_df["test_roc_auc_ci_high"] -
    plot_df["test_roc_auc_ci_low"]
)

plt.figure(figsize=(7, 5))

plt.scatter(
    plot_df["test_events"],
    plot_df["roc_auc_ci_width"],
)

for _, r in plot_df.iterrows():
    plt.text(
        r["test_events"],
        r["roc_auc_ci_width"],
        r["cohort"],
        fontsize=8,
    )

plt.xlabel("Number of test events")
plt.ylabel("ROC-AUC CI width")
plt.title("Event count and uncertainty of model performance")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 25. RISK GROUP PLOTS FOR BEST MODELS
# ============================================================

best_keys = best_summary[
    [
        "workflow",
        "cohort",
        "outcome",
        "feature_set",
        "model_name",
    ]
].drop_duplicates()

risk_best = risk_groups_df.merge(
    best_keys,
    on=[
        "workflow",
        "cohort",
        "outcome",
        "feature_set",
        "model_name",
    ],
    how="inner",
)

order = ["Low", "Intermediate", "High"]

for keys, group in risk_best.groupby(
    ["workflow", "cohort", "outcome", "feature_set", "model_name"]
):
    workflow_name, cohort_name, outcome_col, feature_set, model_name = keys
    
    group = group.copy()
    group["risk_group"] = pd.Categorical(
        group["risk_group"],
        categories=order,
        ordered=True,
    )
    group = group.sort_values("risk_group")
    
    plt.figure(figsize=(6, 4))
    
    plt.bar(
        group["risk_group"].astype(str),
        group["observed_event_rate"],
    )
    
    plt.plot(
        group["risk_group"].astype(str),
        group["mean_predicted_risk"],
        marker="o",
    )
    
    plt.ylabel("Observed event rate / predicted risk")
    plt.xlabel("Risk group")
    plt.title(
        f"{workflow_name}\n{cohort_name} | {outcome_col}"
    )
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 26. FEATURE IMPORTANCE FOR FINAL REFITTED BEST MODELS
# ============================================================

feature_importance_rows = []

for key, obj in best_models.items():

    workflow_name, cohort_name, outcome_col = key

    if "final_pipe" not in obj:
        continue

    pipe = obj["final_pipe"]
    X_test = obj["X_test_final"]
    y_test = obj["y_test"]

    try:
        result = permutation_importance(
            pipe,
            X_test,
            y_test,
            n_repeats=20,
            random_state=42,
            scoring="average_precision",
            n_jobs=-1,
        )

        for feature, mean_imp, std_imp in zip(
            X_test.columns,
            result.importances_mean,
            result.importances_std,
        ):
            feature_importance_rows.append({
                "workflow": workflow_name,
                "cohort": cohort_name,
                "outcome": outcome_col,
                "feature_set": obj["feature_set"],
                "model_name": obj["model_name"],
                "feature": feature,
                "importance_mean": mean_imp,
                "importance_std": std_imp,
            })

    except Exception as e:
        print("Permutation importance failed:", key, e)

feature_importance_df = pd.DataFrame(feature_importance_rows)

display(
    feature_importance_df
    .sort_values("importance_mean", ascending=False)
    .head(50)
)

In [ ]:
# ============================================================
# 27. FINAL INTERPRETATION TABLE
# ============================================================

interpretation = best_summary.copy()

interpretation["roc_auc_ci_width"] = (
    interpretation["test_roc_auc_ci_high"] -
    interpretation["test_roc_auc_ci_low"]
)

interpretation["pr_auc_ci_width"] = (
    interpretation["test_pr_auc_ci_high"] -
    interpretation["test_pr_auc_ci_low"]
)

interpretation["pr_auc_lift_over_event_rate"] = (
    interpretation["test_pr_auc"] /
    interpretation["test_event_rate"]
)

interpretation["roc_auc_interpretation"] = pd.cut(
    interpretation["test_roc_auc"],
    bins=[0, 0.6, 0.7, 0.8, 1.0],
    labels=[
        "Weak",
        "Moderate",
        "Good",
        "Very good",
    ],
    include_lowest=True,
)

interpretation["uncertainty_interpretation"] = pd.cut(
    interpretation["roc_auc_ci_width"],
    bins=[0, 0.15, 0.25, 1.0],
    labels=[
        "Relatively stable",
        "Moderate uncertainty",
        "High uncertainty",
    ],
    include_lowest=True,
)

display(
    interpretation[
        [
            "workflow",
            "cohort",
            "outcome",
            "feature_set",
            "model_name",
            "n_features_original",
            "test_n",
            "test_events",
            "test_event_rate",
            "test_roc_auc",
            "test_roc_auc_ci_low",
            "test_roc_auc_ci_high",
            "roc_auc_interpretation",
            "test_pr_auc",
            "pr_auc_lift_over_event_rate",
            "test_brier",
            "test_calibration_slope",
            "uncertainty_interpretation",
        ]
    ].sort_values(["workflow", "cohort", "outcome"])
)

In [ ]:
# ============================================================
# 28. SAVE OUTPUTS
# ============================================================

results_path = OUTPUT_06B / "06b_supervised_temporal_validation_results.xlsx"
predictions_path = OUTPUT_06B / "06b_supervised_predictions.parquet"

predictions_df.to_parquet(
    predictions_path,
    index=False,
)

with pd.ExcelWriter(results_path) as writer:
    results_df.to_excel(
        writer,
        sheet_name="all_models",
        index=False,
    )
    
    best_summary.to_excel(
        writer,
        sheet_name="best_models",
        index=False,
    )
    
    best_by_feature_set.to_excel(
        writer,
        sheet_name="best_by_feature_set",
        index=False,
    )
    
    best_by_feature_set_refit.to_excel(
        writer,
        sheet_name="best_by_feature_set_refit",
        index=False,
    )

    best_by_feature_set_refit_predictions.to_excel(
        writer,
        sheet_name="feature_set_refit_preds",
        index=False,
    )

    model_comparison.to_excel(
        writer,
        sheet_name="clinical_vs_labs",
        index=False,
    )
    
    risk_groups_df.to_excel(
        writer,
        sheet_name="risk_groups",
        index=False,
    )
    
    interpretation.to_excel(
        writer,
        sheet_name="interpretation_summary",
        index=False,
    )
    
    gap_summary.to_excel(
        writer,
        sheet_name="overfitting_by_model",
        index=False,
    )
    
    feature_importance_df.to_excel(
        writer,
        sheet_name="feature_importance",
        index=False,
    )

print("Saved:")
print(results_path)
print(predictions_path)

In [ ]:
# ============================================================
# 29. FINAL NOTE
# ============================================================

print("""
Notebook 06b completed.

This refactored supervised modelling notebook evaluates two clinically distinct
temporal workflows:

1. early_0_3_prediction
   Early hospitalization prediction using reduced clinical and biomarker
   trajectory summaries from days 0–3.

2. full_0_7_characterization
   First-week trajectory characterization using reduced clinical and biomarker
   trajectory summaries from days 0–7.

The pipeline preserves temporal validation, uses RobustScaler for numerical
features, and prioritizes clinical interpretability, reduced dimensionality,
calibration, and generalization.
""")